# 3. Fine-tune Jina reranker — kênh `jina`

Đây là kênh mạnh nhất đang có (R@5 riêng = 0.8617) và **đã từng được fine-tune một lần** — checkpoint đó (`burst_pairwise_state.pt`, 197 MB) bị loại khỏi gói ref nên notebook khởi tạo lại từ trọng số HuggingFace gốc. Nghĩa là epoch đầu nhiều khả năng còn thua baseline cache; đừng hoảng, hãy đọc cột `channel_only` trong history.

Nếu bạn còn giữ `burst_pairwise_state.pt` ở đâu đó, upload kèm vào `results/jina_reranker/` rồi đặt `INIT_CHECKPOINT = "auto"` — nó sẽ fine-tune *tiếp* từ đó thay vì làm lại từ đầu, điểm xuất phát tốt hơn hẳn.

Đây cũng là notebook chậm nhất: cross-encoder phải chạy qua từng cặp (câu hỏi, đoạn văn) thay vì mã hoá độc lập.

---

## Cách chạy để không mất kết quả

Notebook được cắt thành từng cell nhỏ, **mỗi epoch một cell**. Sau mỗi epoch,
kết quả được ghi ngay xuống `/kaggle/working/jina/` và đóng thành
`jina_results.zip` — nên nếu session chết ở epoch 3 thì epoch 1–2 vẫn còn
nguyên.

Hai chế độ chạy, chọn theo tình huống:

1. **Save Version → Save & Run All (Commit)** — chạy headless, đóng trình duyệt
   được, Kaggle tự lưu toàn bộ `/kaggle/working` thành output. Đây là cách an
   toàn nhất trước chuyện mất mạng hay ngắt server.
2. **Chạy tương tác** — chạy từng cell, và **tải `jina_results.zip` về sau mỗi
   epoch** qua panel Output bên phải. Ở chế độ này `/kaggle/working` sẽ mất khi
   session chết, nên đừng để dồn tới cuối mới tải.

Nếu phải chạy lại: **cứ chạy lại từ cell 1**. Notebook tự đọc `history.json` cũ,
bỏ qua những epoch đã ghi và nạp lại `best_state.pt`, nên không tốn lại thời
gian GPU của phần đã xong. Muốn làm lại sạch thì đặt `RESUME = False`.

**Notebook settings cần bật: GPU và Internet.**

## Cell 1 — Cấu hình

`REF` là dataset dữ liệu + cache (~600 MB, gần như không bao giờ đổi).
`CODE` là nơi chứa 5 file `.py`. Nếu bạn tách code thành một dataset nhỏ riêng
(~160 KB) thì mỗi lần sửa code chỉ upload lại chỗ đó; nếu không có, notebook
tự lùi về `REF/fine_tune` — nên cả hai kiểu đều chạy được, không cần sửa gì.

In [ ]:
import os, sys, json, time
from pathlib import Path

# Giảm phân mảnh bộ nhớ GPU — chính thông báo OOM của PyTorch gợi ý.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REF  = Path("/kaggle/input/fine-tune-ref/ref")   # dataset nhtclone/fine-tune-ref
CODE = Path("/kaggle/input/fine-tune-code")      # dataset code riêng (tuỳ chọn)
WORK = Path("/kaggle/working")

if not CODE.exists():
    CODE = REF / "fine_tune"                     # code nằm luôn trong dataset ref

EPOCHS          = 3
RESUME          = True     # False = bỏ qua history/checkpoint cũ, chạy lại từ đầu
LR              = 1e-05
BATCH_SIZE      = 4        # số truy vấn mỗi bước
ACCUM           = 4        # gradient accumulation
NEGATIVES       = 7        # hard negative mỗi truy vấn mỗi bước
NEGATIVE_DEPTH  = 48       # lấy negative sâu bao nhiêu trong pool lexical đã cache
MAX_LENGTH      = 512
EVAL_BATCH_SIZE = 16
PRECISION       = "auto"   # auto | fp16 | bf16 | fp32 -- xem cell 4
MAX_GPUS        = 0        # 1 = tắt DataParallel (nhẹ VRAM hơn); 0 = dùng hết GPU
TRAIN_TOP_LAYERS = 0      # N block trên cùng + head; 0 = toàn bộ (xem cell 4)
PASSAGES_PER_DOC_TRAIN = 1   # lúc train; lúc đánh giá luôn là 2 như pipeline gốc
TRAIN_QUERIES   = None     # None = dùng cả 1050 truy vấn
SEED            = 2026
INIT_CHECKPOINT = "auto"   # "auto" = tiếp tục từ burst_pairwise_state.pt nếu có
LOSS            = "pairwise"

assert REF.exists(), f"Không thấy {REF} — kiểm tra lại dataset dữ liệu đã attach chưa"
assert (CODE / "burst_common.py").exists(), f"Không thấy code trong {CODE}"
sys.path.insert(0, str(CODE))
os.environ["BURST_ROOT"] = str(REF)
os.environ["BURST_WORK"] = str(WORK)
WORK.mkdir(parents=True, exist_ok=True)
print("dữ liệu:", REF)
print("code   :", CODE)

import torch
print("torch", torch.__version__, "| CUDA:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Bật GPU trong Notebook settings rồi chạy lại"

# Kaggle ghim notebook vào phiên bản dataset lúc attach: upload New Version KHÔNG
# tự cập nhật notebook đang mở. Dòng này bắt lỗi đó ngay thay vì để nó nổ ở cell 4.
import burst_common as _bc, torch_common as _tc
_bc.check_code_version()

## Cell 2 — Dựng lại tập đánh giá 600 truy vấn từ cache

Chỉ CPU, khoảng 1–3 phút. Cell này chạy trước phần GPU có chủ đích: nếu đường dẫn hay cache có vấn đề thì hỏng ở đây, trước khi tốn thời gian GPU.

Con số baseline in ra phải đúng bằng **recall 0.9511 / F2 0.6144** — đó là bản đã nộp.

In [ ]:
import burst_common as bc
import torch_common as tc

documents = bc.DocumentStore(REF / bc.DATA_SUBDIR, preload=True)
bundle    = bc.build_eval_bundle(REF, documents)

baseline, _, _ = bc.lobo_evaluate(bundle)
print(f"\nBASELINE (toàn bộ 6 kênh từ cache)")
print(f"  recall    = {baseline['recall']:.4f}   <- bản đã nộp: 0.9511")
print(f"  precision = {baseline['precision']:.4f}")
print(f"  f2        = {baseline['f2']:.4f}   <- bản đã nộp: 0.6144")
for name, b in baseline["blocks"].items():
    print(f"  block {name}: recall={b['recall']:.4f} f2={b['f2']:.4f}")

## Cell 3 — Tập huấn luyện

1.050 truy vấn có nhãn, đã có sẵn cache retrieval, **không giao** với 600 truy vấn đánh giá. Hard negative lấy thẳng từ pool lexical đã cache — không chạy lại retrieval.

`RunRecorder` chọn `best_state.pt` theo **epoch tốt nhất của chính lần chạy này** trên holdout 600 truy vấn: epoch đầu luôn được lưu, epoch sau chỉ ghi đè khi validate tăng. Việc có vượt baseline cache hay không là một câu hỏi khác và được trả lời riêng ở cột `vượt baseline` trong bảng tổng kết — nó không còn quyết định chuyện lưu hay không, vì baseline của kênh `jina` đến từ một checkpoint không nằm trong gói ref.

In [ ]:
examples = bc.build_train_pool(REF, exclude=bundle.all_ids,
                               negatives=NEGATIVE_DEPTH, limit=TRAIN_QUERIES,
                               seed=SEED, documents=documents)

recorder = bc.RunRecorder(WORK, "jina", baseline, resume=RESUME)
print("Ghi kết quả vào:", recorder.dir)

## Cell 4 — Nạp model

Tải trọng số từ HuggingFace (cần bật Internet). Nếu đang resume, trọng số tốt
nhất của lần chạy trước sẽ được nạp đè.

Nếu Kaggle cấp **2× T4**, model được bọc `nn.DataParallel` để mỗi batch chia
đôi cho hai GPU. Chọn DataParallel chứ không phải DDP vì notebook chạy theo
từng cell, mà DDP thì phải dựng process group — phiền hơn nhiều so với cái nó
đổi lại. Nhược điểm quen thuộc của DataParallel (gom output về GPU 0) gần như
không dính ở đây: cross-encoder trả về một logit mỗi chuỗi, bi-encoder trả về
một vector 1024 chiều, nên thứ chạy ngược về chỉ vài KB còn activation thì nằm
yên tại chỗ.

### Kiểm tra độ chính xác số học

T4 (compute capability 7.5) **không có bf16**, nên autocast rơi về fp16 — dải
số chỉ tới 65504. Model được huấn luyện ở bf16 (dải ~1e38) có thể sinh
activation vượt ngưỡng đó, thành `inf`, rồi `F.normalize` biến `inf/inf` thành
`NaN` và loss là NaN ngay từ bước đầu mà log không nói gì.

Cell này chạy thử **một batch** rồi quyết định: nếu fp16 cho kết quả không hữu
hạn, nó tự chuyển sang fp32. fp32 chậm hơn khoảng 2 lần và tốn bộ nhớ hơn,
nhưng đổi lấy một epoch không NaN thì quá rẻ. Ép tay bằng
`PRECISION = "fp32"` ở cell 1.

In [ ]:
import finetune_jina as ft

model, tokenizer = ft.load_model(REF, ft.DEFAULT_MODEL, INIT_CHECKPOINT,
                                 gradient_checkpointing=True)

if RESUME:
    recorder.load_best_state(model)

print(f"tham số: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

device, amp_dtype, n_gpus = tc.setup(SEED)
if MAX_GPUS:
    n_gpus = min(n_gpus, MAX_GPUS)
model.to(device)

# Chạy thử 1 batch: fp16 có tràn số trên model này không?
probe_texts = [documents[d] for d in examples[0].negatives[:4]]
amp_dtype = tc.choose_precision(model, tokenizer, probe_texts, device, amp_dtype,
                                MAX_LENGTH, kind="cross",
                                requested=PRECISION)

# Pooling có phân biệt được hai đoạn văn khác nhau không? Nếu không thì kênh này
# không thể xếp hạng, và train nó chỉ đẩy token embedding ra vô cực.
tc.check_pooling_discriminates(model, tokenizer, probe_texts, device, amp_dtype,
                               MAX_LENGTH, kind="cross",
                               label="jina")
tc.freeze_lower_layers(model, TRAIN_TOP_LAYERS)
tc.report_memory_budget(model, n_gpus, device)

model = tc.wrap_parallel(model, n_gpus)     # 2x T4 -> mỗi batch chia đôi

## Cell 5 — Optimizer, scheduler, và hai hàm chạy

`train_one_epoch(n)` huấn luyện một epoch. `evaluate_and_record(n)` chấm lại
**chỉ kênh `jina`**, ráp vào LTR fusion 6 kênh, áp ngưỡng động α=0.15,
rồi ghi xuống đĩa ngay.

`scale_for_gpus` nhân `batch_size` lên và chia `accum` xuống theo số GPU, nên
**effective batch không đổi** — hai GPU chỉ rút ngắn thời gian chứ không làm
lệch phép toán huấn luyện so với chạy một GPU. `eval_batch_size` thì cứ nhân
lên vì không có bước optimizer nào cần giữ tương đương.

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    root=REF, work=WORK, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
    accum=ACCUM, negatives=NEGATIVES, negative_depth=NEGATIVE_DEPTH,
    max_length=MAX_LENGTH, eval_batch_size=EVAL_BATCH_SIZE,
    passages_per_doc=PASSAGES_PER_DOC_TRAIN, train_queries=TRAIN_QUERIES,
    warmup_ratio=.1, weight_decay=.01, max_grad_norm=1.0, seed=SEED,
    gradient_checkpointing=True, keep_every_epoch=False, resume=RESUME,
    eval_before_training=False, precision=PRECISION,
    train_top_layers=TRAIN_TOP_LAYERS, max_gpus=MAX_GPUS,
)
args.loss = LOSS
args.temperature = 1.0

tc.scale_for_gpus(args, n_gpus)     # giữ nguyên effective batch, chỉ chia việc

sampler = tc.GroupSampler(examples, documents, NEGATIVES,
                          PASSAGES_PER_DOC_TRAIN, seed=SEED)
groups_per_epoch = (len(examples) + args.batch_size - 1) // args.batch_size
total_steps = max(1, EPOCHS * groups_per_epoch // args.accum)

optimizer = tc.make_optimizer(tc.unwrap(model), LR, args.weight_decay)
scheduler = tc.make_scheduler(optimizer, total_steps, args.warmup_ratio)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))

print(f"{len(examples)} truy vấn, {groups_per_epoch} nhóm/epoch, "
      f"{total_steps} bước optimizer cho {EPOCHS} epoch")


def train_one_epoch(epoch):
    if recorder.already_done(epoch):
        print(f"epoch {epoch} đã ghi từ lần chạy trước — bỏ qua")
        return None
    started = time.perf_counter()
    loss = ft.train_epoch(model, tokenizer, sampler, args, optimizer, scheduler,
                          scaler, device, amp_dtype, epoch)
    print(f"epoch {epoch}: loss={loss:.4f} ({(time.perf_counter()-started)/60:.1f} phút)")
    return loss


def evaluate_and_record(epoch, loss=None):
    if recorder.already_done(epoch):
        print(f"epoch {epoch} đã ghi từ lần chạy trước — bỏ qua")
        return
    started = time.perf_counter()
    questions = {q: bundle.queries[q][0] for q in bundle.all_ids}
    scores = tc.score_cross_encoder(
        model, tokenizer, questions, bundle.extended, documents, bundle.all_ids,
        device, amp_dtype, max_length=MAX_LENGTH, batch_size=EVAL_BATCH_SIZE,
        label="jina")
    metrics, ranked, predictions = bc.lobo_evaluate(
        bundle, {"jina": scores},
        {"jina": bc.rank_by(bundle.extended, scores)})
    metrics["channel_only"] = bc.channel_metrics(bundle, scores)
    improved = recorder.consider(epoch, metrics, tc.unwrap(model).state_dict(),
                                 ranked, predictions, extra={"train_loss": loss})
    if improved:
        recorder.save_channel_scores(scores)
    recorder.package()          # zip nhỏ, tải về được ngay
    print(f"  kênh `jina` riêng: R@5={metrics['channel_only']['Recall@5']:.4f} "
          f"(cache gốc: {bc.channel_metrics(bundle, bundle.scores['jina'])['Recall@5']:.4f})")
    print(f"  chấm điểm mất {(time.perf_counter()-started)/60:.1f} phút")

## Cell 6 — Epoch 1

Chạy xong nhớ tải `jina_results.zip` về nếu đang chạy tương tác.

In [ ]:
loss = train_one_epoch(1)
evaluate_and_record(1, loss)

## Cell 7 — Epoch 2

Chạy xong nhớ tải `jina_results.zip` về nếu đang chạy tương tác.

In [ ]:
loss = train_one_epoch(2)
evaluate_and_record(2, loss)

## Cell 8 — Epoch 3

Epoch cuối.

In [ ]:
loss = train_one_epoch(3)
evaluate_and_record(3, loss)

## Cell 9 — Tổng kết

Bảng lịch sử đầy đủ và danh sách file để tải về.

In [ ]:
history = json.loads((recorder.dir / "history.json").read_text(encoding="utf-8"))

print(f"{'epoch':>9s} {'recall':>8s} {'prec':>7s} {'f2':>7s} {'kênh riêng':>11s}"
      f"  {'đã lưu':>7s}  vượt baseline")
for row in history["history"]:
    channel_only = row.get("channel_only", {}).get("Recall@5")
    print(f"{str(row['epoch']):>9s} {row['recall']:8.4f} {row['precision']:7.4f} "
          f"{row['f2']:7.4f} {(f'{channel_only:.4f}' if channel_only else '-'):>11s}  "
          f"{('BEST' if row.get('improved') else ''):>7s}  "
          f"{'YES' if row.get('beats_baseline') else ''}")

print(f"\nEpoch tốt nhất theo holdout 600 truy vấn: {history['best_epoch']}")
if history["best_epoch"] is None:
    print("Chưa epoch nào được chấm điểm — chưa có gì để lưu.")
elif not history.get("best_beats_baseline"):
    print("Trọng số epoch này ĐÃ được lưu vào best_state.pt, nhưng vẫn dưới "
          "baseline cache — bản nộp vẫn nên dùng điểm cache gốc.")
else:
    print("Epoch này vừa tốt nhất trong lần chạy, vừa vượt baseline cache.")
print("\nNgưỡng nhiễu của bài này là std 0.008 trên Recall. Chênh lệch nhỏ hơn "
      "khoảng đó không phải bằng chứng của gì cả — dự án đã có 4 lần "
      "'thắng CV, thua leaderboard thật'.")

print("\nFile trong /kaggle/working:")
for path in sorted(WORK.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(WORK)}  ({path.stat().st_size/2**20:.1f} MB)")

## Cell 10 — Lưu model đã fine-tune để tải về

`RunRecorder` tự lưu trọng số vào `best_state.pt` ngay từ epoch đầu (xem ghi
chú ở Cell 3), và loại nó khỏi `jina_results.zip` vì nặng (~600 MB+) trong
khi zip kia phải nhỏ để tải nhanh mỗi epoch.

Cell dưới nạp lại đúng epoch tốt nhất từ `best_state.pt` (nếu có), chuyển
sang **fp16** để giảm ~50% dung lượng, rồi lưu ở định dạng HuggingFace chuẩn
(`config.json` + `model.safetensors` + tokenizer) bằng `save_pretrained` vào
`/kaggle/working/jina_finetuned` — dùng lại sau này chỉ cần
`AutoModelForSequenceClassification.from_pretrained(đường_dẫn)`, không cần
nạp `jinaai/jina-reranker-v2-base-multilingual` từ HuggingFace nữa.

In [ ]:
import shutil

# ---- 1) Nạp đúng epoch tốt nhất theo recorder (nếu có) ----
best_state_path = recorder.dir / "best_state.pt"
if best_state_path.exists():
    recorder.load_best_state(model)
    print(f"Đã nạp {best_state_path} (epoch tốt nhất: {history['best_epoch']})")
else:
    print("Không có best_state.pt (chưa epoch nào 'improved') "
          "-- lưu nguyên trọng số hiện tại, tức epoch cuối vừa train.")

base_model = tc.unwrap(model)          # gỡ nn.DataParallel nếu có (2 GPU)

# ---- 2) fp16 để giảm ~50% dung lượng ----
# Đủ chính xác cho inference/tiếp tục fine-tune; nếu cần fp32 tuyệt đối thì bỏ dòng dưới.
base_model = base_model.half()

SAVE_DIR = WORK / "jina_finetuned"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
base_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("\nĐã lưu model (fp16) vào:", SAVE_DIR)
for p in sorted(SAVE_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(SAVE_DIR)}  ({p.stat().st_size/2**20:.1f} MB)")

zip_path = shutil.make_archive(str(WORK / "jina_finetuned"), "zip", root_dir=SAVE_DIR)
print(f"\nZip để tải về qua panel Output: {zip_path}  "
      f"({Path(zip_path).stat().st_size/2**20:.1f} MB)")